In [49]:
:dep pipe-lib = { path = "../pipe-lib" }

In [2]:
:dep reqwest
:dep tokio = { version = "1.39.2", features = ["rt", "time", "rt-multi-thread"] }
:dep bytes

In [3]:
use std::io::copy;
use std::io::Cursor;
use std::fs::File;
use std::path::PathBuf;
use tokio;

pub fn download_binary(url: String, destination: PathBuf) -> Result<PathBuf, String> {
    async fn download(url: String, destination: PathBuf) -> Result<(), String> {
        let client = reqwest::Client::new();
        let response = client.get(url.clone()).send();
        let filename = PathBuf::from(url.clone()).file_name().unwrap().to_str().unwrap().to_string();
        match response.await {
            | Ok(data) => match data.bytes().await {
                | Ok(content) => {
                    let mut output = File::create(destination.join(filename.clone())).unwrap();
                    let _ = copy(&mut Cursor::new(content.clone()), &mut output);
                    debug!(filename = filename, "=> {} Downloaded", Label::output());
                    Ok(())
                }
                | Err(_) => Err(format!("No content downloaded from {url}")),
            },
            | Err(_) => Err(format!("Failed to download {url}")),
        }
    }
    let runtime = tokio::runtime::Builder::new_current_thread().enable_all().build().unwrap();
    let _ = runtime.block_on(download(url.clone(), destination.clone()));
    let filename = PathBuf::from(url.clone()).file_name().unwrap().to_str().unwrap().to_string();
    Ok(destination.join(filename))
}

In [4]:
let url = "https://code.ornl.gov/api/v4/projects/15383/packages/generic/x86_64-unknown-linux-gnu/v0.0.3/pipe".to_string();
let content = download_binary(url, PathBuf::from("/root/dev/notebooks"));
println!("{:?}", content);

Downloaded pipe


In [ ]:
let path = "/tmp/pipe/iopbgzaGbV/pipe";
